# EntoKey Foundation Model v0.6 — 100 000 images

Это уже не 720-image pilot. Notebook строит сбалансированный план из **30k iNaturalist + 30k BIOSCAN-5M + 25k GBIF + 15k DiSSCo**, затем пропускает изображения через DINOv2-base по возобновляемым шардам. Выбери GPU; A100/L4 предпочтительнее T4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, subprocess, sys, torch
assert torch.cuda.is_available(), 'Включи GPU: Runtime → Change runtime type → GPU'
print('GPU:', torch.cuda.get_device_name(0))
REPO_URL = 'https://github.com/SaniyaSani/EntoKey.git'
PROJECT = Path('/content/EntoKey')
STORE = Path('/content/drive/MyDrive/EntoKey/Foundation_v06')
MASTER = STORE/'manifests/master_manifest.parquet'
assert MASTER.exists(), 'Сначала закончи Foundation_Corpus_v05_SETUP_Colab.ipynb'
if not PROJECT.exists(): subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT)], check=True)
else: subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

## 1. Построить строгий 100k training plan
Ячейка проверит, что представлены все четыре источника. Taxa балансируются внутри каждого источника; фотографии одного specimen остаются в одной группе.

In [ ]:
WORK = STORE/'training_100k'
MODEL = STORE/'models_foundation_v06'
subprocess.run([sys.executable, 'scripts/run_foundation_v06.py', '--master-manifest', str(MASTER), '--work-dir', str(WORK), '--model-dir', str(MODEL), '--stage', 'plan'], check=True)
print((WORK/'corpus_report.md').read_text())

## 2. DINOv2-base embeddings — возобновляемо
Один шард ≈ 2 000 изображений. Картинки по URL декодируются в памяти и после DINOv2 не сохраняются; остаются vectors + provenance. Увеличивай `MAX_SHARDS_THIS_SESSION` на A100. Перезапускай ячейку в новых сессиях: готовые шарды будут пропущены.

In [ ]:
MAX_SHARDS_THIS_SESSION = 2  # 0 = попытаться закончить все; для T4 начни с 1–2
subprocess.run([sys.executable, 'scripts/run_foundation_v06.py', '--master-manifest', str(MASTER), '--work-dir', str(WORK), '--model-dir', str(MODEL), '--stage', 'embed', '--max-shards', str(MAX_SHARDS_THIS_SESSION)], check=True)
index = json.loads((WORK/'manifest_shards/shards.json').read_text())
done = len(list((WORK/'embedding_shards').glob('shard_*/complete.json')))
print(f'COMPLETED: {done}/{index["shard_count"]} shards')

## 3. Иерархическая модель и retrieval index
Запусти, когда предыдущая ячейка показывает все шарды. Species heads используют только качество A/B; iNaturalist Research Grade остаётся сильным family/genus материалом, но не автоматически истиной для cryptic species.

In [ ]:
index = json.loads((WORK/'manifest_shards/shards.json').read_text())
done = len(list((WORK/'embedding_shards').glob('shard_*/complete.json')))
assert done == index['shard_count'], f'Сначала закончи embeddings: {done}/{index["shard_count"]}'
subprocess.run([sys.executable, 'scripts/run_foundation_v06.py', '--master-manifest', str(MASTER), '--work-dir', str(WORK), '--model-dir', str(MODEL), '--stage', 'train'], check=True)
print('FOUNDATION MODEL READY:', MODEL)
print((MODEL/'training_report_hierarchical.json').read_text()[:8000])

## 4. Упаковать checkpoint
В архив попадут модель, конфигурация и retrieval index, но не исходные изображения.

In [ ]:
import shutil
archive = shutil.make_archive(str(STORE/'EntoKey_foundation_v06_model'), 'zip', root_dir=MODEL)
print('MODEL ARCHIVE:', archive)